In [1]:
import numpy as np
import torch

In [2]:
X = np.array([160, 170, 180, 190])
y = np.array([0, 0, 1, 1])

print("입력값 X:", X)
print("정답값 y:", y)

입력값 X: [160 170 180 190]
정답값 y: [0 0 1 1]


In [3]:
X_mean = np.mean(X)
X_std = np.std(X)
X_norm = (X - X_mean) / X_std

print(X_mean)
print(X_std)
print(X_norm)

175.0
11.180339887498949
[-1.34164079 -0.4472136   0.4472136   1.34164079]


In [4]:
X_norm_tensor = torch.tensor(X_norm, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

X_norm_tensor = X_norm_tensor.reshape(-1, 1)
y_tensor = y_tensor.reshape(-1, 1)

print(X_norm_tensor)
print(y_tensor)

print(X_norm_tensor.shape)
print(y_tensor.shape)

tensor([[-1.3416],
        [-0.4472],
        [ 0.4472],
        [ 1.3416]])
tensor([[0.],
        [0.],
        [1.],
        [1.]])
torch.Size([4, 1])
torch.Size([4, 1])


In [ ]:
class PerceptronModel(torch.nn.Module):
    def __init__(self):
        # super().__init__() : 부모 class인 torch.nn.Module의 초기화를 먼저 실행
        # 이 줄이 있어야 PyTorch가 self.linear의 weight, bias를
        # '학습 대상 파라미터'로 제대로 등록하고 관리합니다. (반드시 필요)
        super().__init__()

        # self.linear : 이 모델이 가지고 있는 선형 계산 부품
        # torch.nn.Linear(1, 1)은 입력 특성 1개를 받아 출력 1개(H 값)를 내보냅니다.
        # 내부적으로 H(x)=a*X_norm+b 를 계산하며,
        # 기존의 a는 self.linear.weight, b는 self.linear.bias 입니다
        self.linear = torch.nn.Linear(1, 1)

    def forward(self, x):
        # forward : 입력 x가 들어왔을 때 실제 계산 순서를 정의하는 함수
        # 이전 파일에서 학습 루프에 직접 적었던 두 줄이 바로 여기로 들어옴

        # H(x) = a*X_norm+b (sigmoid전의 선형 계산값)
        H = self.linear(x)

        # z = sigmoid(H)
        z = torch.sigmoid(H)
        
        # 이번 파일은 torch.nn.BCELoss()를 쓰므로, H가 아니라 z를 반환
        return z

In [6]:
# torch.manual_seed(42) : 랜덤 초기값 고정. 반드시 model 생성 '전'에 둡니다.
# - model 생성 시 내부의 self.linear.weight(=a), self.linear.bias(=b)가 랜덤 초기화되기 때문
torch.manual_seed(42)

# model = PerceptronModel() : 우리가 정의한 class로 실제 모델 객체를 만듭니다.
# - 이때 __init__이 실행되어 self.linear가 준비됩니다.
# - 이전 파일처럼 linear를 단독으로 만들지 않고, moel 안의 self.linear로 가지고 있습니다.
model = PerceptronModel()

# model 안에 자동으로 만들어진 weight(a=)와 bias(=b)의 '학습 전' 초기값을 확인합니다.
# model.linear.weight 는 기존 강의의 a 역할을 합니다.
# model.linear.bias 는 기존 강의의 b 역할을 합니다.
print("model.linear.weight:", model.linear.weight)
print("model.linear.bias:", model.linear.bias)

# model 전체 구조도 출력해 봅니다. 어떤 부품(self.linear)을 가지고 있는지 보여 줍니다.
print("model 구조:", model)

model.linear.weight: Parameter containing:
tensor([[0.7645]], requires_grad=True)
model.linear.bias: Parameter containing:
tensor([0.8300], requires_grad=True)
model 구조: PerceptronModel(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)


In [7]:
# model(X_norm_tensor)를 호출하면 내부적으로 forward가 실행되어
# H = self.linear(X_norm_tensor)
# z = torch.sigmoid(H)
# 가 계산되고, 그 z가 반환됩니다.
#
# 아직 학습 전이라 weight, bias는 랜덤 초기값 상태입니다. (예측이 맞을 필요는 없습니다.)
# 여기서는 '모델이 올바른 shape의 z를 내놓는가'만 확인합니다.
with torch.no_grad():
    z_test = model(X_norm_tensor)

# z는 각 데이터의 예측 화률ㄹ이므로 y_tensor와 같은 (4, 1) shape이어야 합니다
print("z_test shape:", z_test.shape)
print("z_test:\n", z_test)

z_test shape: torch.Size([4, 1])
z_test:
 tensor([[0.4512],
        [0.6197],
        [0.7635],
        [0.8648]])


In [8]:
# torch.nn.BCELoss()
# - Binary Cross Entropy Cost를 계산해 주는 PyTorch 부품입니다.
# - 사용법: mean_cost = criterion(z, y_tensor) <- H가 아니라 z(sigmoid 통과값)를 넣습니다.
# - 우리 model의 forward가 이미 z를 반환하므로, 그 z를 그대로 넣으면 됩니다.
criterion = torch.nn.BCELoss()

print("criterion 준비 완료:", criterion)

criterion 준비 완료: BCELoss()


In [11]:
# learning_rate(학습률): 한 번에 weight, bias를 얼마나 크게 수정할지 정하는 값입니다.
learning_rate = 0.1

# epochs(에폭): 전체 데이터를 몇 번 반복해서 학습할지 정하는 값입니다.
epochs = 1000

print("learning_rate:", learning_rate)
print("epochs:", epochs)

# torch.optim.SGD(model.parameters(), lr=learning_rate)
# - model.parameters() : optimizer가 업데이트할 학습 대상입니다. 
#                       model 안의 self.linear.weight(=a)와 self.linear.bias(=b)를 가져옵니다.
# - 이전 파일에서는 linear.parameters()를 넘겼지만,
#   이제 linear가 model 안에 있으므로 model.parameters()를 넘깁니다.
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# model.parameters()가 실제로 무엇을 가져오는지 눈으로 확인
# 이 출력 결과에 보이는 값들이 optimizer가 업데이트할 학습 대상
# (첫 번째가 self.linear.weight(=a), 두 번째가 self.linear.bias(=b)입니다)
print(list(model.parameters()))

learning_rate: 0.1
epochs: 1000
[Parameter containing:
tensor([[0.7645]], requires_grad=True), Parameter containing:
tensor([0.8300], requires_grad=True)]


In [15]:
# 한 번의 epoch에서 일어나는 단계 (반드시 이 순서):
#  1. optimizer.zero_grad()       : 이전 epoch의 grad를 0으로 초기화
#  2. z = model(X_norm_tensor)    : forward 실행 (내부에서 H 계산 -> sigmoid -> z 반환)
#  3. mean_cost = criterion(z, y)  : torch.nn.BCELoss 로 Cost 계산 (z를 넣음! H 아님)
#  4. mean_cost.backward()        : model 내부 weight.grad, bias.grad 자동 계산
#  5. optimizer.step()            : model 내부 weight, bias 업데이트
#  6. 학습 상태 출력

for epoch in range(epochs):

    # ---- 1. 이전 epoch에서 계산된 grad 초기화 ----
    # optimizer는 model.parameters()(= self.linear.weight, self.linear.bias)를 관리하므로,
    # 이 한 줄이 model 내부 두 파라미터의 grad를 한꺼번에 0으로 만듭니다.
    optimizer.zero_grad()

    # ---- 2. model 호출 -> 내부적으로 forward 실행 ----
    # z = model(X_norm_tensor) 는 내부적으로
    #     H = self.linear(X_norm_tensor)   (H(x)=a·X_norm+b, 선형 계산값 - 확률 아님)
    #     z = torch.sigmoid(H)             (0~1 사이 예측 확률)
    # 를 실행한 결과입니다. (외부에서 linear를 직접 호출하지 않습니다.)
    z = model(X_norm_tensor)

    # ---- 3. torch.nn.BCELoss 로 Cost 계산 ----
    # 주의: criterion에는 H가 아니라, sigmoid를 통과한 z를 넣습니다.
    #      (우리 forward가 이미 z를 반환하므로 그 z를 그대로 사용합니다.)
    mean_cost = criterion(z, y_tensor)

    # ---- 4. backward: model 내부 파라미터의 grad 자동 계산 ----
    # Cost에서 출발해 model 내부 weight, bias까지 거꾸로 따라가며 미분값을 구해
    #    model.linear.weight.grad (= 기존 grad_a)
    #    model.linear.bias.grad   (= 기존 grad_b)
    # 에 저장합니다.
    mean_cost.backward()

    # ---- 5. optimizer가 model 내부 weight와 bias 업데이트 ----
    optimizer.step()

    # ---- 6. 학습 상태 출력 ----
    # 입력 특성이 1개라 weight, bias에 값이 하나씩만 있으므로 .item()으로 숫자만 꺼냅니다.
    if epoch % 100 == 0 or epoch == epochs - 1:
        print(
            f"epoch={epoch}, "
            f"Cost={mean_cost.item():.6f}, "
            f"weight(a)={model.linear.weight.item():.6f}, "
            f"bias(b)={model.linear.bias.item():.6f}"
        )

    # (참고) 초반 3 epoch에서만 grad 값을 확인해 봅니다.
    # 이전 파일에서는 linear.weight.grad, linear.bias.grad 를 봤지만,
    # 이제 linear가 model 안에 있으므로 model.linear.weight.grad, model.linear.bias.grad 를 봅니다.
    #   기존 grad_a -> model.linear.weight.grad
    #   기존 grad_b -> model.linear.bias.grad
    if epoch < 3:
        print(
            f"    (확인용) model.linear.weight.grad={model.linear.weight.grad.item():.6f}, "
            f"model.linear.bias.grad={model.linear.bias.grad.item():.6f}"
        )

epoch=0, Cost=0.495464, weight(a)=0.793780, bias(b)=0.812529
    (확인용) model.linear.weight.grad=-0.292415, model.linear.bias.grad=0.174793
    (확인용) model.linear.weight.grad=-0.286153, model.linear.bias.grad=0.169918
    (확인용) model.linear.weight.grad=-0.280072, model.linear.bias.grad=0.165171
epoch=100, Cost=0.178670, weight(a)=2.290082, bias(b)=0.173212
epoch=200, Cost=0.125357, weight(a)=3.002210, bias(b)=0.061586
epoch=300, Cost=0.099283, weight(a)=3.509002, bias(b)=0.026837
epoch=400, Cost=0.082901, weight(a)=3.912263, bias(b)=0.013229
epoch=500, Cost=0.071398, weight(a)=4.250606, bias(b)=0.007116
epoch=600, Cost=0.062789, weight(a)=4.543496, bias(b)=0.004091
epoch=700, Cost=0.056068, weight(a)=4.802371, bias(b)=0.002480
epoch=800, Cost=0.050660, weight(a)=5.034644, bias(b)=0.001570
epoch=900, Cost=0.046207, weight(a)=5.245449, bias(b)=0.001031
epoch=999, Cost=0.042507, weight(a)=5.436657, bias(b)=0.000701


In [16]:
# 학습된 weight, bias는 optimizer.step()에 의해 1000번 반복 업데이트된 값입니다.
# 입력 특성이 1개라 값이 하나씩만 있으므로 .item()으로 숫자만 꺼냅니다.
print("학습된 weight(a):", model.linear.weight.item())
print("학습된 bias(b):", model.linear.bias.item())

# tensor 원본 형태도 함께 확인해 둡니다. (shape과 requires_grad 표시를 볼 수 있습니다.)
print("model.linear.weight:", model.linear.weight)
print("model.linear.bias:", model.linear.bias)

# 학습 후 grad도 한 번 확인해 봅니다. (마지막 epoch의 grad가 남아 있습니다.)
#   기존 grad_a -> model.linear.weight.grad
#   기존 grad_b -> model.linear.bias.grad
print("model.linear.weight.grad:", model.linear.weight.grad)
print("model.linear.bias.grad:", model.linear.bias.grad)

학습된 weight(a): 5.436656951904297
학습된 bias(b): 0.0007013267604634166
model.linear.weight: Parameter containing:
tensor([[5.4367]], requires_grad=True)
model.linear.bias: Parameter containing:
tensor([0.0007], requires_grad=True)
model.linear.weight.grad: tensor([[-0.0185]])
model.linear.bias.grad: tensor([2.6396e-05])


In [20]:
# 키가 185cm인 사람이 농구선수인지 예측합니다.
input_height = 185

# 새로운 입력값도 학습 데이터와 '같은 기준'으로 정규화해야 합니다.
# 학습 때 사용한 X_mean, X_std 를 그대로 다시 사용합니다. (새로 계산하면 안 됩니다.)
input_norm = (input_height - X_mean) / X_std

# 예측은 학습이 아니므로 weight, bias를 업데이트하지 않습니다.
# 따라서 미분 계산 기록도 필요 없으므로 with torch.no_grad() 안에서 계산합니다.
with torch.no_grad():
    # torch.nn.Linear(1, 1)에 넣으려면 입력 shape을 (1, 1)로 맞춰야 합니다.
    #   [[input_norm]] : 이중 대괄호로 감싸 (데이터 1개, 입력 특성 1개) = (1, 1) 형태로 만듭니다.
    input_norm_tensor = torch.tensor([[input_norm]], dtype=torch.float32)
    print("input_norm_tensor shape:", input_norm_tensor.shape)

    # z_new = model(input_norm_tensor) 는 내부적으로
    #     H_new = self.linear(input_norm_tensor)   (선형 계산값 - 확률 아님)
    #     z_new = torch.sigmoid(H_new)             (예측 확률 - 0~1 사이)
    # 를 실행한 결과입니다. H_new가 직접 보이지 않을 뿐, 계산은 그대로 일어납니다.
    z_new = model(input_norm_tensor)

    # 0.5 이상이면 1(농구선수), 미만이면 0(농구선수 아님)
    # z_new는 shape (1, 1) tensor이므로 .item()으로 숫자 하나를 꺼냅니다.
    pred = 1 if z_new.item() >= 0.5 else 0

print(f"키가 {input_height}cm인 사람의 농구선수 확률(z): {z_new.item():.4f}")
if pred == 1:
    print("판별 결과: 농구선수입니다.")
else:
    print("판별 결과: 농구선수가 아닙니다.")

input_norm_tensor shape: torch.Size([1, 1])
키가 185cm인 사람의 농구선수 확률(z): 0.9923
판별 결과: 농구선수입니다.
